In [8]:
import pandas as pd
from sqlalchemy import create_engine
from google import genai

In [9]:
username = "root"
password = "Tuktuk000"
host = "localhost"
port = "3306"
database = "customer_behavior"

engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}"
)

print("MySQL connected!")

MySQL connected!


In [10]:
df = pd.read_sql("SELECT * FROM customer LIMIT 5", engine)

df

,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,previous_purchases,payment_method,frequency_of_purchases,age-group,age_group,purchase_frequency_days
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,14,Venmo,Fortnightly,Middle-aged,Middle-aged,14
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,2,Cash,Fortnightly,Young adult,Young adult,14
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,23,Credit Card,Weekly,Middle-aged,Middle-aged,7
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,49,PayPal,Weekly,Young adult,Young adult,7
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,31,PayPal,Annually,Middle-aged,Middle-aged,365


In [ ]:
import os
from google import genai

client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

Gemini client created!


In [12]:
def generate_sql(question):

    prompt = f"""
You are an expert MySQL SQL generator.

We have a MySQL table named `customer`.

The columns are:
- customer_id
- age
- gender
- item_purchased
- category
- purchase_amount
- location
- size
- color
- season
- review_rating
- subscription_status
- shipping_type
- discount_applied
- previous_purchases
- payment_method
- frequency_of_purchases
- age_group
- purchase_frequency_days

Convert the user's question into a MySQL query.

Rules:
1. Use only the `customer` table.
2. Use only the columns listed above.
3. Return ONLY the SQL query.
4. Do not use markdown.
5. Do not explain the query.
6. Always give calculated columns meaningful aliases.

Examples:
SELECT COUNT(*) AS total_customers
SELECT AVG(purchase_amount) AS average_purchase_amount

User question:
{question}
"""

    response = client.interactions.create(
        model="gemini-3.6-flash",
        input=prompt
    )

    return response.output_text.strip()

In [14]:
def ask_customer_ai(question):

    # Generate SQL
    sql = generate_sql(question)

    print("Generated SQL:")
    print(sql)
    print()

    # Safety check
    if not sql.strip().upper().startswith("SELECT"):
        print("❌ Unsafe SQL query.")
        return None

    # Execute SQL
    result = pd.read_sql(sql, engine)

    print("Database Result:")
    print(result)

    return result

In [15]:
question = input("Ask a question about your customers: ")

result = ask_customer_ai(question)

Ask a question about your customers:  How many customers are there?


Generated SQL:
SELECT COUNT(*) AS total_customers FROM customer;

Database Result:
   total_customers
0             3900


In [16]:
def generate_insight(question, result):

    prompt = f"""
You are a retail business analyst.

The user asked:
{question}

The database returned:
{result.to_string(index=False)}

Analyze this result and provide a short business insight.

Rules:
1. Use only the information provided in the database result.
2. Do not invent numbers.
3. Keep the explanation simple.
4. Mention the important finding.
5. If appropriate, mention what the result could mean for the business.
"""

    response = client.interactions.create(
        model="gemini-3.6-flash",
        input=prompt
    )

    return response.output_text.strip()

In [18]:
def generate_local_insight(question, result):

    if result.empty:
        return "No data was found for this question."

    # If result contains category and revenue
    if "category" in result.columns and "total_revenue" in result.columns:

        top_category = result.iloc[0]["category"]
        revenue = result.iloc[0]["total_revenue"]

        return (
            f"{top_category} generated the highest revenue "
            f"of ${revenue:,.2f}."
        )

    # If result contains customer count
    if "total_customers" in result.columns:

        total = result.iloc[0]["total_customers"]

        return f"There are {total:,} customers in the database."

    return "The query was successfully executed and returned the result."



In [20]:
query = """
SELECT category,
       SUM(purchase_amount) AS total_revenue
FROM customer
GROUP BY category
ORDER BY total_revenue DESC
LIMIT 1;
"""

result = pd.read_sql(query, engine)

print(result)

   category  total_revenue
0  Clothing       104264.0


In [21]:
def generate_local_insight(result):

    if result.empty:
        return "No data found."

    category = result.iloc[0]["category"]
    revenue = result.iloc[0]["total_revenue"]

    return f"{category} generated the highest revenue with ${revenue:,.2f}."

In [22]:
print(generate_local_insight(result))

Clothing generated the highest revenue with $104,264.00.


In [ ]:
question = "How many customers are there?"

result = ask_customer_ai(question)

if result is not None:
    insight = generate_local_insight(question, result)

    print("\nBusiness Insight:")
    print(insight)

In [ ]:
question = "Which category has the highest total revenue?"

result = ask_customer_ai(question)

if result is not None:
    insight = generate_insight(question, result)
    print("\nAI Business Insight:")
    print(insight)